In [1]:
# Entity Type Extension for FinDoc Validator
# Adding: AMOUNT, DATE, ACCOUNT, SSN, FORM entities
# Strategy: Create synthetic training data + fine-tune existing model

import torch
import pandas as pd
import numpy as np
import re
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
import random
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

print("🚀 EXTENDING ENTITY TYPES FOR FINANCIAL DOCUMENTS")
print("="*60)
print("Adding: AMOUNT, DATE, ACCOUNT, SSN, FORM entities")
print("Strategy: Synthetic data generation + continued fine-tuning")

🚀 EXTENDING ENTITY TYPES FOR FINANCIAL DOCUMENTS
Adding: AMOUNT, DATE, ACCOUNT, SSN, FORM entities
Strategy: Synthetic data generation + continued fine-tuning


In [2]:
# ==========================================
# STEP 1: LOAD YOUR EXISTING MODEL
# ==========================================

print("\n📂 LOADING YOUR TRAINED MODEL...")

# Load your saved model
model_path = "./models/financial-ner-v1"
try:
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForTokenClassification.from_pretrained(model_path)
    print(f"✅ Loaded your trained model from {model_path}")
except:
    print("⚠️  Model not found, loading base model...")
    model_name = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # We'll create a new model with extended labels


📂 LOADING YOUR TRAINED MODEL...
✅ Loaded your trained model from ./models/financial-ner-v1


In [3]:
# ==========================================
# STEP 2: DEFINE EXTENDED LABEL SYSTEM
# ==========================================

print("\n🏷️  DEFINING EXTENDED LABEL SYSTEM...")

# Extended labels for financial documents
extended_labels = {
    'O': 0,          # Outside
    'PER_B': 1,      # Person - Beginning 
    'PER_I': 2,      # Person - Inside
    'LOC_B': 3,      # Location - Beginning
    'LOC_I': 4,      # Location - Inside  
    'ORG_B': 5,      # Organization - Beginning
    'ORG_I': 6,      # Organization - Inside
    # NEW FINANCIAL ENTITIES:
    'AMOUNT_B': 7,   # Dollar amounts, percentages ($5000, 15.5%)
    'AMOUNT_I': 8,   
    'DATE_B': 9,     # Dates (March 15, 2024, 12/01/2023)
    'DATE_I': 10,
    'ACCOUNT_B': 11, # Account numbers (123456789, routing numbers)
    'ACCOUNT_I': 12,
    'SSN_B': 13,     # Social Security Numbers (123-45-6789)
    'SSN_I': 14,
    'FORM_B': 15,    # Form types (1040, W-2, Form 8829)
    'FORM_I': 16
}

id2label = {v: k for k, v in extended_labels.items()}
num_labels = len(extended_labels)

print(f"✅ Extended to {num_labels} entity types:")
for label, id in extended_labels.items():
    if id >= 7:
        print(f"   🆕 {label}")


🏷️  DEFINING EXTENDED LABEL SYSTEM...
✅ Extended to 17 entity types:
   🆕 AMOUNT_B
   🆕 AMOUNT_I
   🆕 DATE_B
   🆕 DATE_I
   🆕 ACCOUNT_B
   🆕 ACCOUNT_I
   🆕 SSN_B
   🆕 SSN_I
   🆕 FORM_B
   🆕 FORM_I


In [4]:
# ==========================================
# STEP 3: SYNTHETIC DATA GENERATION
# ==========================================

print("\n🎭 GENERATING SYNTHETIC TRAINING DATA...")

def generate_amounts():
    """Generate realistic financial amounts"""
    amounts = []
    
    # Dollar amounts
    for _ in range(50):
        amount = random.choice([
            f"${random.randint(100, 999999):,}",
            f"${random.randint(1, 999)}.{random.randint(10, 99)}",
            f"${random.randint(1000, 99999):,}.{random.randint(10, 99)}"
        ])
        amounts.append(amount)
    
    # Percentages
    for _ in range(20):
        pct = f"{random.uniform(0.1, 99.9):.1f}%"
        amounts.append(pct)
    
    return amounts

def generate_dates():
    """Generate realistic date formats"""
    dates = []
    
    # Different date formats
    formats = [
        lambda d: d.strftime("%B %d, %Y"),      # March 15, 2024
        lambda d: d.strftime("%m/%d/%Y"),       # 03/15/2024
        lambda d: d.strftime("%m-%d-%Y"),       # 03-15-2024
        lambda d: d.strftime("%Y-%m-%d"),       # 2024-03-15
        lambda d: d.strftime("%B %d"),          # March 15
        lambda d: d.strftime("%m/%d/%y"),       # 03/15/24
    ]
    
    # Generate dates from 2020-2025
    start_date = datetime(2020, 1, 1)
    end_date = datetime(2025, 12, 31)
    
    for _ in range(100):
        random_date = start_date + timedelta(
            days=random.randint(0, (end_date - start_date).days)
        )
        format_func = random.choice(formats)
        dates.append(format_func(random_date))
    
    return dates

def generate_accounts():
    """Generate realistic account numbers"""
    accounts = []
    
    # Bank account numbers (8-12 digits)
    for _ in range(30):
        length = random.randint(8, 12)
        account = ''.join([str(random.randint(0, 9)) for _ in range(length)])
        accounts.append(account)
    
    # Routing numbers (9 digits)
    for _ in range(20):
        routing = ''.join([str(random.randint(0, 9)) for _ in range(9)])
        accounts.append(routing)
    
    return accounts

def generate_ssns():
    """Generate realistic (fake) SSN formats"""
    ssns = []
    
    for _ in range(30):
        # XXX-XX-XXXX format
        part1 = ''.join([str(random.randint(0, 9)) for _ in range(3)])
        part2 = ''.join([str(random.randint(0, 9)) for _ in range(2)])
        part3 = ''.join([str(random.randint(0, 9)) for _ in range(4)])
        ssn = f"{part1}-{part2}-{part3}"
        ssns.append(ssn)
    
    return ssns

def generate_forms():
    """Generate tax form numbers and types"""
    forms = [
        "Form 1040", "1040", "Form 1040EZ", "1040EZ",
        "Form W-2", "W-2", "Form W-4", "W-4",
        "Form 1099", "1099", "Form 1099-MISC", "1099-MISC",
        "Form 8829", "8829", "Schedule C", "Schedule A",
        "Form 941", "941", "Form 990", "990"
    ]
    return forms

# Generate all entity examples
amounts = generate_amounts()
dates = generate_dates()
accounts = generate_accounts()
ssns = generate_ssns()
forms = generate_forms()

print(f"✅ Generated synthetic data:")
print(f"   💰 {len(amounts)} amount examples")
print(f"   📅 {len(dates)} date examples")
print(f"   🏦 {len(accounts)} account examples") 
print(f"   🔢 {len(ssns)} SSN examples")
print(f"   📄 {len(forms)} form examples")


🎭 GENERATING SYNTHETIC TRAINING DATA...
✅ Generated synthetic data:
   💰 70 amount examples
   📅 100 date examples
   🏦 50 account examples
   🔢 30 SSN examples
   📄 20 form examples


In [5]:
def create_labels_for_sentence(sentence, entities):
    """Create BIO labels for a sentence using tokenizer offsets"""
    # Map entity type to label prefix
    label_mapping = {
        'person': 'PER',
        'organization': 'ORG',
        'amount': 'AMOUNT',
        'date': 'DATE',
        'account': 'ACCOUNT',
        'ssn': 'SSN',
        'form': 'FORM'
    }
    
    # Find all entity spans (start, end char positions and type)
    entity_spans = []
    for entity_type, entity_value in entities.items():
        if entity_type in label_mapping and entity_value.strip():
            start_idx = sentence.find(entity_value)
            if start_idx != -1:
                end_idx = start_idx + len(entity_value)
                entity_spans.append((start_idx, end_idx, label_mapping[entity_type]))
    
    # Tokenize with offsets
    tokenized = tokenizer(sentence, return_offsets_mapping=True, add_special_tokens=False)
    tokens = tokenized.tokens() if hasattr(tokenized, 'tokens') else tokenizer.convert_ids_to_tokens(tokenized['input_ids'])
    offsets = tokenized['offset_mapping']
    
    labels = ['O'] * len(tokens)
    
    for start, end, prefix in entity_spans:
        # Find tokens whose offsets overlap with the entity span
        first = True
        for i, (tok_start, tok_end) in enumerate(offsets):
            if tok_end <= start:
                continue
            if tok_start >= end:
                break
            if first:
                labels[i] = f"{prefix}_B"
                first = False
            else:
                labels[i] = f"{prefix}_I"
    
    return tokens, labels

In [6]:
# ==========================================
# STEP 5: PREPARE FOR CONTINUED TRAINING
# ==========================================

print("\n⚙️ PREPARING EXTENDED MODEL FOR TRAINING...")

# Create new model with extended labels if needed
if model.config.num_labels != num_labels:
    print(f"🔄 Resizing model from {model.config.num_labels} to {num_labels} labels...")
    
    # Create new model with extended labels
    from transformers import AutoConfig
    
    config = AutoConfig.from_pretrained(
        "distilbert-base-uncased",
        num_labels=num_labels,
        id2label=id2label,
        label2id=extended_labels
    )
    
    new_model = AutoModelForTokenClassification.from_pretrained(
        "distilbert-base-uncased",
        config=config
    )
    
    # If you had a previously trained model, you could copy some weights here
    # For now, we'll start fresh with the extended labels
    model = new_model
    print("✅ Created new model with extended labels")

# Convert synthetic data to dataset format
def convert_to_hf_dataset(sentences, labels):
    """Convert our synthetic data to HuggingFace dataset format"""
    
    # Convert label strings to IDs
    label_ids = []
    for sentence_labels in labels:
        ids = [extended_labels[label] for label in sentence_labels]
        label_ids.append(ids)
    
    return Dataset.from_dict({
        'gold_token': sentences,
        'gold_label': label_ids
    })

def create_synthetic_sentences():
    """Create realistic sentences with multiple entity types"""
    
    # Sentence templates for different document types
    bank_templates = [
        "{person} deposited {amount} into account {account} on {date}.",
        "Account {account} processed a withdrawal of {amount} on {date}.",
        "{organization} transferred {amount} to {person} on {date}.",
        "The account balance for {account} was {amount} as of {date}.",
        "{person} opened account {account} with {organization} on {date}."
    ]
    
    tax_templates = [
        "{person} filed {form} reporting income of {amount} on {date}.",
        "The taxpayer's Social Security Number {ssn} is required for {form}.",
        "{person} with SSN {ssn} owes {amount} according to {form}.",
        "{form} must be filed by {date} for the tax year.",
        "Income of {amount} was reported on {form} by {person}."
    ]
    
    loan_templates = [
        "{person} applied for a {amount} loan with {organization} on {date}.",
        "The loan application {form} requires SSN {ssn} and income verification.",
        "{organization} approved {person} for {amount} on {date}.",
        "Account {account} will receive loan proceeds of {amount}.",
        "{person} submitted {form} to {organization} on {date}."
    ]
    
    # Sample names and organizations (reuse from your trained model)
    persons = ["John Smith", "Sarah Johnson", "Michael Brown", "Lisa Davis", "David Wilson"]
    organizations = ["Bank of America", "Wells Fargo", "JPMorgan Chase", "Citibank", "Goldman Sachs"]
    
    sentences = []
    labels = []
    
    # Generate sentences from each template type
    for templates in [bank_templates, tax_templates, loan_templates]:
        for _ in range(20):  # 20 sentences per template type
            template = random.choice(templates)
            
            # Fill template with random entities
            sentence_data = {}
            if "{person}" in template:
                sentence_data["person"] = random.choice(persons)
            if "{organization}" in template:
                sentence_data["organization"] = random.choice(organizations)
            if "{amount}" in template:
                sentence_data["amount"] = random.choice(amounts)
            if "{date}" in template:
                sentence_data["date"] = random.choice(dates)
            if "{account}" in template:
                sentence_data["account"] = random.choice(accounts)
            if "{ssn}" in template:
                sentence_data["ssn"] = random.choice(ssns)
            if "{form}" in template:
                sentence_data["form"] = random.choice(forms)
            
            try:
                sentence = template.format(**sentence_data)
                tokens, token_labels = create_labels_for_sentence(sentence, sentence_data)
                
                sentences.append(tokens)
                labels.append(token_labels)
            except KeyError:
                continue  # Skip if template has missing keys
    
    return sentences, labels

synthetic_sentences, synthetic_labels = create_synthetic_sentences()

synthetic_dataset = convert_to_hf_dataset(synthetic_sentences, synthetic_labels)

print(f"✅ Created synthetic dataset: {len(synthetic_dataset)} examples")

print("\n🎯 READY FOR EXTENDED ENTITY TRAINING!")
print("""
NEXT STEPS:
1. Tokenize synthetic data (same process as before)
2. Configure training for extended entities
3. Fine-tune model on new entity types
4. Test on financial document samples

This will give your FinDoc Validator the ability to detect:
💰 Dollar amounts and percentages
📅 Dates in multiple formats  
🏦 Account and routing numbers
🔢 Social Security Numbers
📄 Tax form numbers and types
""")


⚙️ PREPARING EXTENDED MODEL FOR TRAINING...
🔄 Resizing model from 7 to 17 labels...


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Created new model with extended labels
✅ Created synthetic dataset: 60 examples

🎯 READY FOR EXTENDED ENTITY TRAINING!

NEXT STEPS:
1. Tokenize synthetic data (same process as before)
2. Configure training for extended entities
3. Fine-tune model on new entity types
4. Test on financial document samples

This will give your FinDoc Validator the ability to detect:
💰 Dollar amounts and percentages
📅 Dates in multiple formats  
🏦 Account and routing numbers
🔢 Social Security Numbers
📄 Tax form numbers and types



In [7]:
# ==========================================
# STEP 6: TOKENIZATION FOR EXTENDED ENTITIES
# ==========================================

print("\n🔧 TOKENIZING SYNTHETIC DATA...")

def tokenize_and_align_labels_extended(examples):
    """Same tokenization process, but with extended label set"""
    tokenized_inputs = tokenizer(
        examples["gold_token"], 
        truncation=True, 
        is_split_into_words=True,
        padding=True
    )
    
    labels = []
    for i, label in enumerate(examples["gold_label"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        
        labels.append(label_ids)
    
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Tokenize synthetic dataset
tokenized_synthetic = synthetic_dataset.map(tokenize_and_align_labels_extended, batched=True)

print("✅ Tokenization complete!")
print(f"📊 Ready to train on {len(tokenized_synthetic)} extended examples")

print("""
🚀 YOUR EXTENDED MODEL IS READY!

To start training:
1. Set up training arguments
2. Create trainer with extended model
3. Run trainer.train() 
4. Test on financial documents with new entity types!

This will complete your FinDoc Validator's core AI capabilities!
""")


🔧 TOKENIZING SYNTHETIC DATA...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

✅ Tokenization complete!
📊 Ready to train on 60 extended examples

🚀 YOUR EXTENDED MODEL IS READY!

To start training:
1. Set up training arguments
2. Create trainer with extended model
3. Run trainer.train() 
4. Test on financial documents with new entity types!

This will complete your FinDoc Validator's core AI capabilities!



In [8]:
# DETAILED: Creating Trainer with Extended Model
# This is the missing piece that bridges your 7-label model to 17-label model
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorForTokenClassification,
    AutoConfig
)
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

print("🔧 EXTENDING YOUR TRAINED MODEL TO HANDLE NEW ENTITY TYPES")
print("="*60)

# ==========================================
# STEP 1: LOAD YOUR EXISTING 7-LABEL MODEL
# ==========================================

print("📂 Loading your existing trained model...")

# Your current model (7 labels: O, PER_B, PER_I, LOC_B, LOC_I, ORG_B, ORG_I)
try:
    existing_model = AutoModelForTokenClassification.from_pretrained("./models/financial-ner-v1")
    existing_tokenizer = AutoTokenizer.from_pretrained("./models/financial-ner-v1")
    print(f"✅ Loaded existing model with {existing_model.config.num_labels} labels")
except:
    print("⚠️  Using base model for demonstration")
    existing_model = AutoModelForTokenClassification.from_pretrained("distilbert-base-uncased", num_labels=7)
    existing_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

print(f"Current model output shape: {existing_model.classifier.out_features} neurons")

# ==========================================
# STEP 2: CREATE EXTENDED MODEL ARCHITECTURE  
# ==========================================

print("\n🏗️  CREATING EXTENDED MODEL ARCHITECTURE...")

# Define the extended label system
extended_labels = {
    'O': 0, 'PER_B': 1, 'PER_I': 2, 'LOC_B': 3, 'LOC_I': 4, 'ORG_B': 5, 'ORG_I': 6,
    'AMOUNT_B': 7, 'AMOUNT_I': 8, 'DATE_B': 9, 'DATE_I': 10, 
    'ACCOUNT_B': 11, 'ACCOUNT_I': 12, 'SSN_B': 13, 'SSN_I': 14,
    'FORM_B': 15, 'FORM_I': 16
}

id2label = {v: k for k, v in extended_labels.items()}
num_labels = len(extended_labels)

print(f"🎯 Target: {num_labels} labels")
print("New entity types:", [label for label in extended_labels.keys() if extended_labels[label] >= 7])

# Create new model configuration
config = AutoConfig.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,           # 17 labels instead of 7
    id2label=id2label,
    label2id=extended_labels
)

print(f"✅ Created config for {config.num_labels} labels")

# ==========================================
# STEP 3: MODEL WEIGHT TRANSFER (THE MAGIC!)
# ==========================================

print("\n🎭 TRANSFERRING LEARNED KNOWLEDGE...")

# Create the extended model
extended_model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    config=config
)

print(f"New model output shape: {extended_model.classifier.out_features} neurons")

# CRITICAL: Copy the weights from your trained model!
# This preserves all the knowledge about PER/ORG/LOC entities
if hasattr(existing_model, 'classifier'):
    print("🔄 Transferring learned weights...")
    
    with torch.no_grad():
        # Copy the BERT layers (these learned language understanding)
        extended_model.distilbert.load_state_dict(existing_model.distilbert.state_dict())
        print("✅ Copied BERT encoder weights")
        
        # Copy the classification head for existing labels (0-6)
        old_classifier_weights = existing_model.classifier.weight.data[:7]  # First 7 labels
        old_classifier_bias = existing_model.classifier.bias.data[:7]
        
        # Transfer to new model
        extended_model.classifier.weight.data[:7] = old_classifier_weights
        extended_model.classifier.bias.data[:7] = old_classifier_bias
        print("✅ Copied classification weights for PER/ORG/LOC entities")
        
        # New entity weights (7-16) are randomly initialized
        print("🎲 New entity types (AMOUNT/DATE/ACCOUNT/SSN/FORM) have random weights - will be learned!")

print("\n🧠 WHAT JUST HAPPENED:")
print("""
Your model now has:
- ✅ Preserved knowledge: PER/ORG/LOC detection (from your 98% accuracy training)
- 🎲 New capabilities: AMOUNT/DATE/ACCOUNT/SSN/FORM detection (will learn from synthetic data)
- 🚀 Best of both worlds: Keep existing performance + add new features
""")

# ==========================================
# STEP 4: SETUP EXTENDED TRAINING
# ==========================================

print("\n⚙️ CONFIGURING TRAINING FOR EXTENDED MODEL...")

# Set device (MPS for M3)
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda") 
else:
    device = torch.device("cpu")

extended_model.to(device)
print(f"🔧 Model moved to {device}")

# Training arguments - adjusted for continued learning
training_args = TrainingArguments(
    output_dir="./models/financial-ner-extended",
    num_train_epochs=100,                    # Fewer epochs since we're fine-tuning
    per_device_train_batch_size=16,        # Smaller batch size for stability
    per_device_eval_batch_size=16,
    learning_rate=2e-5,                    # Lower learning rate to preserve existing knowledge
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs/extended",
    logging_steps=50,
    eval_strategy="no",
    # eval_steps=200,
    # save_steps=400,
    # load_best_model_at_end=True,
    # metric_for_best_model="eval_loss",
    dataloader_num_workers=0,
    report_to=None,
)

print("✅ Training configuration created")
print(f"   📊 Epochs: {training_args.num_train_epochs}")
print(f"   📈 Learning rate: {training_args.learning_rate}")
print(f"   💾 Output: {training_args.output_dir}")

# Data collator for padding
data_collator = DataCollatorForTokenClassification(
    tokenizer=existing_tokenizer,  # Same tokenizer
    padding=True
)

# ==========================================
# STEP 5: DEFINE EXTENDED EVALUATION METRICS
# ==========================================

from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

def compute_extended_metrics(eval_pred):
    """Compute entity-level metrics (precision, recall, F1) for extended labels."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    # Convert ids to labels, ignoring special tokens (-100)
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Compute overall metrics
    precision = precision_score(true_labels, true_predictions)
    recall = recall_score(true_labels, true_predictions)
    f1 = f1_score(true_labels, true_predictions)

    # Detailed report per entity type
    report = classification_report(true_labels, true_predictions, digits=4)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "report": report
    }


# ==========================================
# STEP 6: CREATE THE EXTENDED TRAINER
# ==========================================

print("\n👨‍🏫 CREATING TRAINER FOR EXTENDED MODEL...")

def create_extended_trainer(train_dataset, eval_dataset=None):
    """Create trainer for the extended model"""
    
    trainer = Trainer(
        model=extended_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=existing_tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_extended_metrics,
    )
    
    return trainer

print("✅ Extended trainer creation function ready!")


# ==========================================
# STEP 6B: ENTITY RECONSTRUCTION HELPER
# ==========================================

def postprocess_entities(entities):
    cleaned = []
    for text, label in entities:
        # Rule: long digit-only strings are ACCOUNT, not FORM
        if label == "FORM" and text.isdigit() and len(text) >= 6:
            cleaned.append((text, "ACCOUNT"))
        else:
            cleaned.append((text, label))
    return cleaned


def reconstruct_entities(tokens, labels):
    """Reconstruct entity spans from token-level predictions"""
    entities = []
    current_entity = []
    current_label = None
    
    for token, label in zip(tokens, labels):
        # Skip special tokens
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue

        # Merge subwords (e.g., "##ing")
        if token.startswith("##"):
            if current_entity:
                current_entity[-1] = current_entity[-1] + token[2:]
            else:
                # orphaned subword, treat as normal token
                current_entity.append(token)
        else:
            if label.endswith("_B"):
                # Save previous entity
                if current_entity:
                    entities.append((" ".join(current_entity), current_label))
                current_entity = [token]
                current_label = label.split("_")[0]
            elif label.endswith("_I") and current_label == label.split("_")[0]:
                current_entity.append(token)
            else:
                # Outside or inconsistent I-tag
                if current_entity:
                    entities.append((" ".join(current_entity), current_label))
                    current_entity = []
                current_label = None

    # Catch last entity
    if current_entity:
        entities.append((" ".join(current_entity), current_label))

    return entities



# ==========================================
# STEP 7: TRAINING PROCESS EXPLANATION
# ==========================================

print("\n🎓 UNDERSTANDING THE TRAINING PROCESS:")
print("""
When you call trainer.train(), here's what happens:

1. 📖 FORWARD PASS:
   - Synthetic sentence: "John Smith deposited $5000 on March 15, 2024"
   - Model predicts: [PER_B, PER_I, O, AMOUNT_B, O, DATE_B, DATE_I, DATE_I]
   
2. 📊 LOSS CALCULATION:
   - Existing entities (John Smith): Model already knows these well (low loss)
   - New entities ($5000, March 15, 2024): Model learning from scratch (higher loss)
   
3. 🔄 BACKPROPAGATION:
   - Updates weights for new entity types
   - Preserves (mostly) existing entity knowledge
   - Gradually improves new entity detection

4. 📈 CONVERGENCE:
   - Existing entities: Stay at ~95% accuracy
   - New entities: Improve from ~20% to ~85% accuracy
   - Overall system becomes more capable!
""")

# ==========================================
# STEP 8: USAGE EXAMPLE
# ==========================================

print("\n🚀 HOW TO USE THIS:")
print("""
# Assuming you have your synthetic dataset ready:

# 1. Create the trainer
trainer = create_extended_trainer(
    train_dataset=tokenized_synthetic_dataset,
    eval_dataset=tokenized_validation_dataset  # Optional
)

# 2. Start training
print("🔥 Starting extended entity training...")
trainer.train()

# 3. Test the results
# Now your model will detect ALL entity types!
""")

print("\n💡 KEY INSIGHTS:")
print("""
✅ PRESERVATION: Your 98% PER/ORG/LOC accuracy is mostly preserved
🎯 EXTENSION: Model learns AMOUNT/DATE/ACCOUNT/SSN/FORM from synthetic data  
🚀 EFFICIENCY: Much faster than training from scratch
🎓 LEARNING: You understand both transfer learning AND synthetic data generation
""")

print("\nThis is the missing piece that makes your FinDoc Validator truly powerful! 🎉")


🔧 EXTENDING YOUR TRAINED MODEL TO HANDLE NEW ENTITY TYPES
📂 Loading your existing trained model...
✅ Loaded existing model with 7 labels
Current model output shape: 7 neurons

🏗️  CREATING EXTENDED MODEL ARCHITECTURE...
🎯 Target: 17 labels
New entity types: ['AMOUNT_B', 'AMOUNT_I', 'DATE_B', 'DATE_I', 'ACCOUNT_B', 'ACCOUNT_I', 'SSN_B', 'SSN_I', 'FORM_B', 'FORM_I']
✅ Created config for 17 labels

🎭 TRANSFERRING LEARNED KNOWLEDGE...


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


New model output shape: 17 neurons
🔄 Transferring learned weights...
✅ Copied BERT encoder weights
✅ Copied classification weights for PER/ORG/LOC entities
🎲 New entity types (AMOUNT/DATE/ACCOUNT/SSN/FORM) have random weights - will be learned!

🧠 WHAT JUST HAPPENED:

Your model now has:
- ✅ Preserved knowledge: PER/ORG/LOC detection (from your 98% accuracy training)
- 🎲 New capabilities: AMOUNT/DATE/ACCOUNT/SSN/FORM detection (will learn from synthetic data)
- 🚀 Best of both worlds: Keep existing performance + add new features


⚙️ CONFIGURING TRAINING FOR EXTENDED MODEL...
🔧 Model moved to mps
✅ Training configuration created
   📊 Epochs: 100
   📈 Learning rate: 2e-05
   💾 Output: ./models/financial-ner-extended

👨‍🏫 CREATING TRAINER FOR EXTENDED MODEL...
✅ Extended trainer creation function ready!

🎓 UNDERSTANDING THE TRAINING PROCESS:

When you call trainer.train(), here's what happens:

1. 📖 FORWARD PASS:
   - Synthetic sentence: "John Smith deposited $5000 on March 15, 2024"
   -

In [9]:
trainer = create_extended_trainer(
    train_dataset=tokenized_synthetic,
    eval_dataset=None
)

trainer.train()

/var/folders/vh/7zvptz6n65g83c7dq3r7p1k00000gn/T/ipykernel_8002/2913040251.py:203: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/opt/anaconda3/envs/findoc-validator/lib/python3.10/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,2.573200
100,0.385000
150,0.021800
200,0.010100
250,0.007400
300,0.006200
350,0.005600
400,0.005300


TrainOutput(global_step=400, training_loss=0.3768249655514955, metrics={'train_runtime': 61.6828, 'train_samples_per_second': 97.272, 'train_steps_per_second': 6.485, 'total_flos': 68917782420000.0, 'train_loss': 0.3768249655514955, 'epoch': 100.0})

In [10]:
test_sentence = "John Smith deposited $5000 into account 123456789 on March 15, 2024"

# Pick device: use MPS if available, else fallback
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Move model to device
extended_model.to(device)

# Tokenize
inputs = existing_tokenizer(test_sentence, return_tensors="pt").to(device)

# Run inference
with torch.no_grad():
    outputs = extended_model(**inputs).logits
    predictions = torch.argmax(outputs, dim=2)

# Convert predictions back to labels
tokens = existing_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu())
pred_labels = [id2label[p.item()] for p in predictions[0].cpu()]

# Reconstruct entities
entities = reconstruct_entities(tokens, pred_labels)
entities = postprocess_entities(entities)
print("Detected Entities:", entities)

Detected Entities: [('john smith', 'PER'), ('$ 5000', 'AMOUNT'), ('123456789', 'ACCOUNT'), ('march 15 , 2024', 'DATE')]
